# Experiment: Deepfake Robustness Full Standalone Colab Run

This notebook is a fully standalone Colab implementation of the project. It does **not** import the repository files from disk. Instead, it embeds the project logic directly into notebook cells so you can upload only this notebook plus your datasets and still run the full workflow.

What is included directly in the notebook:
- configuration and runtime setup
- dataset preparation and cleaning
- BioBERT detector training and inference helpers
- SeqGAN generator, discriminator, rollout, and trainer
- BioGPT adversarial rewrite agent with optional LoRA fine-tuning
- adversarial training loop
- evaluation metrics, saved plots, and notebook-side result analysis
- live Weights & Biases login and sync

Expected Colab workflow:
1. Switch runtime to `GPU`.
2. Upload this notebook.
3. Upload your real and fake dataset files.
4. Update the dataset paths in the configuration cell.
5. Run all cells from top to bottom.


## Environment Setup

This first code cell installs the required Python packages inside Colab. `bitsandbytes` is installed as an optional dependency so the adversarial BioGPT agent can use 4-bit loading when the runtime supports it.

If Colab asks for a runtime restart after package installation, restart once and then run the notebook again from the top.


In [ ]:
import subprocess
import sys

INSTALL_BITSANDBYTES = True

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.0.0',
    'transformers>=4.38.0',
    'datasets>=2.18.0',
    'accelerate>=0.27.0',
    'scikit-learn>=1.3.0',
    'pandas>=2.0.0',
    'numpy>=1.24.0',
    'tqdm>=4.65.0',
    'wandb>=0.16.0',
    'sentencepiece>=0.1.99',
    'protobuf>=3.20.0',
    'pyyaml>=6.0.1',
    'peft>=0.11.0',
    'evaluate>=0.4.1',
    'bert-score>=0.3.13',
    'safetensors>=0.4.2',
    'matplotlib>=3.8.0',
    'seaborn>=0.13.2',
    'python-dotenv>=1.0.1'
])

if INSTALL_BITSANDBYTES:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes>=0.43.1'])
        print('Installed bitsandbytes successfully.')
    except subprocess.CalledProcessError as exc:
        print(f'bitsandbytes installation failed; the notebook will continue without 4-bit agent loading. Details: {exc}')

print('Package installation complete.')


## Notebook Configuration

This cell replaces the repository YAML config with an in-notebook configuration dictionary. The values are aligned with the project files, but everything lives inside the notebook so no external config file is required.

Update the dataset paths here before you run the training pipeline. The fake dataset path can point to a single supported file, a directory, or a `.zip` archive containing CSV, JSON, JSONL, or TXT files.


In [ ]:
from __future__ import annotations

import copy
import gc
import hashlib
import html
import io
import json
import logging
import math
import os
import platform
import random
import re
import unicodedata
import zipfile
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import wandb
import yaml
from IPython.display import Image, display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.distributions import Categorical
from torch.optim import Adam, AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    get_linear_schedule_with_warmup,
)

WORK_DIR = Path('/content/deepfake_robustness_standalone').resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_HOME = '/content/hf_cache'
PIP_CACHE_DIR = '/content/pip_cache'
TMPDIR = '/content/tmp'
for directory in [HF_HOME, PIP_CACHE_DIR, TMPDIR]:
    Path(directory).mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = HF_HOME
os.environ['TRANSFORMERS_CACHE'] = HF_HOME
os.environ['PIP_CACHE_DIR'] = PIP_CACHE_DIR
os.environ['TMPDIR'] = TMPDIR
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

CONFIG: dict[str, Any] = {
    'project': {
        'name': 'deepfake_robustness',
        'output_dir': 'experiments',
        'seed': 42,
        'num_workers': 2,
        'device': 'auto',
    },
    'data': {
        'pubmed_path': '/content/pubmed_real.csv',
        'fake_path': '/content/med_mmhl',
        'processed_path': 'data/processed',
        'train_file': 'train.csv',
        'val_file': 'val.csv',
        'test_file': 'test.csv',
        'text_columns': ['abstract', 'text', 'content'],
        'title_column': 'title',
        'label_column': 'label',
        'positive_label': 1,
        'negative_label': 0,
        'min_words': 50,
        'max_length': 512,
        'tokenizer_name': 'dmis-lab/biobert-base-cased-v1.2',
        'use_model_tokenizer': True,
        'train_split': 0.70,
        'val_split': 0.15,
        'test_split': 0.15,
    },
    'seqgan': {
        'vocab_size': 10000,
        'embed_dim': 64,
        'hidden_dim': 512,
        'num_layers': 2,
        'seq_len': 200,
        'pretrain_epochs_g': 100,
        'pretrain_epochs_d': 50,
        'adversarial_epochs': 200,
        'rollout_num': 16,
        'batch_size': 32,
        'lr_g': 0.0001,
        'lr_d': 0.0001,
        'generated_pool_size': 10000,
        'temperature': 1.0,
        'gradient_clip_norm': 5.0,
        'checkpoint_dir': 'models/seqgan/checkpoints',
    },
    'detector': {
        'model_name': 'dmis-lab/biobert-base-cased-v1.2',
        'num_labels': 2,
        'max_length': 384,
        'batch_size': 4,
        'grad_accum_steps': 4,
        'epochs_per_round': 5,
        'lr': 0.00002,
        'weight_decay': 0.01,
        'warmup_ratio': 0.1,
        'fp16': True,
        'gradient_checkpointing': True,
        'checkpoint_dir': 'models/detector/checkpoints',
    },
    'agent': {
        'model_name': 'microsoft/biogpt',
        'checkpoint_dir': 'agents/checkpoints',
        'evasion_threshold': 0.5,
        'high_conf_threshold': 0.8,
        'max_new_tokens': 192,
        'temperature': 0.9,
        'top_p': 0.95,
        'lora_r': 8,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'finetune_epochs': 3,
        'batch_size': 2,
        'grad_accum_steps': 4,
        'lr': 0.0001,
        'max_length': 384,
        'gradient_checkpointing': True,
        'use_4bit': True,
    },
    'loop': {
        'num_rounds': 10,
        'fake_pool_size': 1000,
        'hard_sample_top_k': 200,
        'save_checkpoint_every': 2,
        'round_data_dir': 'experiments/round_data',
    },
    'evaluation': {
        'metrics_path': 'experiments/metrics',
        'plots_path': 'experiments/plots',
        'log_predictions': True,
        'save_plots': True,
        'threshold': 0.5,
    },
    'tracking': {
        'use_wandb': True,
        'wandb_project': 'deepfake_robustness',
        'wandb_entity': '',
        'wandb_mode': 'online',
        'wandb_dir': 'experiments/wandb',
        'log_tables': True,
        'log_images': True,
    },
}

ANALYSIS_CONFIG = {
    'threshold_grid': [round(value, 2) for value in np.linspace(0.05, 0.95, 19)],
    'top_error_examples': 10,
    'top_rewrite_examples': 10,
    'generated_text_preview_count': 5,
    'display_saved_plots': True,
    'max_saved_plot_images': 12,
}

CONFIG['agent']['use_4bit'] = bool(torch.cuda.is_available())
CONFIG


## Runtime Validation And W&B Login

This cell checks that the uploaded dataset paths exist, prints the hardware summary, and performs a real Weights & Biases login. The notebook uses online sync by default, matching your request.


In [ ]:
real_path = Path(CONFIG['data']['pubmed_path'])
fake_path = Path(CONFIG['data']['fake_path'])
missing_paths = [str(path) for path in [real_path, fake_path] if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Upload your datasets into Colab and update CONFIG["data"]["pubmed_path"] / CONFIG["data"]["fake_path"] first. Missing: '
        + ', '.join(missing_paths)
    )

print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

wandb.login(relogin=True)
os.environ['WANDB_PROJECT'] = CONFIG['tracking']['wandb_project']
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_NOTEBOOK_NAME'] = 'run.ipynb'
if CONFIG['tracking']['wandb_entity']:
    os.environ['WANDB_ENTITY'] = CONFIG['tracking']['wandb_entity']
else:
    os.environ.pop('WANDB_ENTITY', None)

print('Weights & Biases login completed.')


## Shared Utilities And Experiment Logger

This cell defines the shared helpers that the rest of the notebook uses. These functions replace the utility and logging modules from the original repository and are intentionally kept compatible with the same configuration structure.


In [ ]:
LOGGER = logging.getLogger('deepfake_robustness_notebook')

def configure_logging(level: int = logging.INFO) -> None:
    logging.basicConfig(
        level=level,
        format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
        force=True,
    )

def resolve_path(path: str | Path) -> Path:
    candidate = Path(path)
    if candidate.is_absolute():
        return candidate
    return WORK_DIR / candidate

def ensure_dir(path: str | Path) -> Path:
    directory = resolve_path(path)
    directory.mkdir(parents=True, exist_ok=True)
    return directory

def save_json(data: Dict[str, Any], path: str | Path) -> None:
    destination = resolve_path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('w', encoding='utf-8') as handle:
        json.dump(data, handle, indent=2, sort_keys=True)

def load_json(path: str | Path) -> Dict[str, Any]:
    with resolve_path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def save_yaml(data: Dict[str, Any], path: str | Path) -> None:
    destination = resolve_path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(data, handle, sort_keys=False)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device(device_name: str = 'auto') -> torch.device:
    if device_name != 'auto':
        return torch.device(device_name)
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def to_serializable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {key: to_serializable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_serializable(item) for item in value]
    return value

def maybe_autocast(enabled: bool, device: torch.device):
    if not enabled or device.type != 'cuda':
        return nullcontext()
    return torch.cuda.amp.autocast()

def count_trainable_parameters(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def get_env_or_default(name: str, default: Any = None) -> Any:
    value = os.getenv(name)
    if value is None or value == '':
        return default
    return value

class ExperimentLogger:
    def __init__(self, config: dict, run_name: str, job_type: str, tags: Optional[Iterable[str]] = None) -> None:
        tracking_config = config.get('tracking', {})
        self.enabled = bool(tracking_config.get('use_wandb', False))
        self.log_tables = bool(tracking_config.get('log_tables', True))
        self.log_images = bool(tracking_config.get('log_images', True))
        self.run = None
        if not self.enabled:
            return
        try:
            wandb_dir = ensure_dir(tracking_config.get('wandb_dir', 'experiments/wandb'))
            self.run = wandb.init(
                project=get_env_or_default('WANDB_PROJECT', tracking_config.get('wandb_project')),
                entity=get_env_or_default('WANDB_ENTITY', tracking_config.get('wandb_entity') or None),
                mode=get_env_or_default('WANDB_MODE', tracking_config.get('wandb_mode', 'online')),
                dir=str(wandb_dir),
                name=run_name,
                job_type=job_type,
                tags=list(tags or []),
                config=to_serializable(config),
                reinit=True,
            )
        except Exception as exc:
            LOGGER.warning('W&B init failed, continuing without it: %s', exc)
            self.enabled = False
            self.run = None

    def log_metrics(self, metrics: Dict[str, Any], step: Optional[int] = None, prefix: Optional[str] = None) -> None:
        if not self.run:
            return
        payload = {}
        for key, value in metrics.items():
            if isinstance(value, (int, float)):
                payload[f'{prefix}/{key}' if prefix else key] = value
        if payload:
            self.run.log(payload, step=step)

    def log_dataframe(self, name: str, frame: pd.DataFrame, max_rows: int = 250) -> None:
        if not self.run or not self.log_tables:
            return
        try:
            limited = frame.head(max_rows).copy()
            self.run.log({name: wandb.Table(dataframe=limited)})
        except Exception as exc:
            LOGGER.warning('Failed to log table %s to W&B: %s', name, exc)

    def log_images_from_paths(self, images: Dict[str, Path]) -> None:
        if not self.run or not self.log_images:
            return
        try:
            payload = {name: wandb.Image(str(path)) for name, path in images.items() if Path(path).exists()}
            if payload:
                self.run.log(payload)
        except Exception as exc:
            LOGGER.warning('Failed to log images to W&B: %s', exc)

    def finish(self) -> None:
        if self.run is not None:
            self.run.finish()
            self.run = None

configure_logging()
set_seed(CONFIG['project']['seed'])
save_yaml(CONFIG, 'runtime_config.yaml')
print(f'Working directory: {WORK_DIR}')


## Data Preparation Pipeline

This cell recreates the repository data pipeline directly in the notebook. It supports CSV, JSON, JSONL, TXT, directory trees, and `.zip` archives so you can work comfortably with Colab uploads.


In [ ]:
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
WHITESPACE_PATTERN = re.compile(r'\s+')

def discover_text_column(frame: pd.DataFrame, candidates: Sequence[str]) -> str:
    for column in candidates:
        if column in frame.columns:
            return column
    raise ValueError(f'No text column found in columns={list(frame.columns)}')

def normalize_text(text: str) -> str:
    cleaned = html.unescape(str(text))
    cleaned = HTML_TAG_PATTERN.sub(' ', cleaned)
    cleaned = unicodedata.normalize('NFKC', cleaned)
    cleaned = WHITESPACE_PATTERN.sub(' ', cleaned).strip()
    return cleaned

def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def truncate_text(text: str, max_length: int, tokenizer: Optional[AutoTokenizer] = None) -> str:
    if tokenizer is None:
        words = text.split()
        return ' '.join(words[:max_length])
    token_ids = tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=max_length)
    return tokenizer.decode(token_ids, skip_special_tokens=True)

def _load_text_file_from_bytes(raw_bytes: bytes) -> pd.DataFrame:
    text = raw_bytes.decode('utf-8', errors='ignore')
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return pd.DataFrame({'text': lines})

def _load_file_like(name: str, raw_bytes: bytes) -> pd.DataFrame:
    suffix = Path(name).suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(io.BytesIO(raw_bytes))
    if suffix == '.json':
        return pd.read_json(io.BytesIO(raw_bytes))
    if suffix == '.jsonl':
        return pd.read_json(io.BytesIO(raw_bytes), lines=True)
    if suffix == '.txt':
        return _load_text_file_from_bytes(raw_bytes)
    raise ValueError(f'Unsupported source format: {name}')

def load_source(path: Path) -> pd.DataFrame:
    if path.is_dir():
        frames: List[pd.DataFrame] = []
        for child in sorted(path.iterdir()):
            if child.is_dir() or child.suffix.lower() in {'.csv', '.json', '.jsonl', '.txt', '.zip'}:
                frames.append(load_source(child))
        if not frames:
            raise FileNotFoundError(f'No supported source files found under {path}')
        return pd.concat(frames, ignore_index=True)
    suffix = path.suffix.lower()
    if suffix == '.zip':
        frames: List[pd.DataFrame] = []
        with zipfile.ZipFile(path, 'r') as archive:
            for member in sorted(archive.namelist()):
                member_path = Path(member)
                if member_path.suffix.lower() in {'.csv', '.json', '.jsonl', '.txt'} and not member.endswith('/'):
                    with archive.open(member) as handle:
                        frames.append(_load_file_like(member, handle.read()))
        if not frames:
            raise FileNotFoundError(f'No supported files found inside archive: {path}')
        return pd.concat(frames, ignore_index=True)
    return _load_file_like(path.name, path.read_bytes())

def standardize_frame(frame: pd.DataFrame, text_candidates: Sequence[str], label: int, title_column: str) -> pd.DataFrame:
    text_column = discover_text_column(frame, text_candidates)
    standardized = pd.DataFrame()
    standardized['text'] = frame[text_column].astype(str)
    standardized['title'] = frame[title_column].astype(str) if title_column in frame.columns else ''
    standardized['label'] = label
    return standardized

def clean_frame(frame: pd.DataFrame, min_words: int, max_length: int, tokenizer: Optional[AutoTokenizer] = None) -> pd.DataFrame:
    cleaned = frame.copy()
    cleaned['text'] = cleaned['text'].fillna('').map(normalize_text)
    cleaned = cleaned[cleaned['text'].astype(bool)]
    cleaned['word_count'] = cleaned['text'].map(lambda value: len(value.split()))
    cleaned = cleaned[cleaned['word_count'] >= min_words]
    cleaned['text'] = cleaned['text'].map(lambda value: truncate_text(value, max_length=max_length, tokenizer=tokenizer))
    cleaned['text_hash'] = cleaned['text'].map(text_hash)
    cleaned = cleaned.drop_duplicates(subset='text_hash')
    cleaned = cleaned.drop(columns=['word_count', 'text_hash'])
    cleaned = cleaned.reset_index(drop=True)
    return cleaned

def load_tokenizer_if_enabled(config: dict) -> Optional[AutoTokenizer]:
    if not config['data'].get('use_model_tokenizer', False):
        return None
    tokenizer_name = config['data'].get('tokenizer_name') or config['detector']['model_name']
    try:
        LOGGER.info('Loading tokenizer %s for truncation.', tokenizer_name)
        return AutoTokenizer.from_pretrained(tokenizer_name)
    except Exception as exc:
        LOGGER.warning('Falling back to whitespace truncation because tokenizer loading failed: %s', exc)
        return None

def split_frame(frame: pd.DataFrame, train_ratio: float, val_ratio: float, test_ratio: float, seed: int) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6:
        raise ValueError('train/val/test splits must sum to 1.0')
    train_frame, temp_frame = train_test_split(frame, test_size=(1.0 - train_ratio), random_state=seed, stratify=frame['label'])
    relative_test_ratio = test_ratio / (val_ratio + test_ratio)
    val_frame, test_frame = train_test_split(temp_frame, test_size=relative_test_ratio, random_state=seed, stratify=temp_frame['label'])
    return train_frame.reset_index(drop=True), val_frame.reset_index(drop=True), test_frame.reset_index(drop=True)

def prepare_data(config: dict) -> dict:
    data_config = config['data']
    set_seed(config['project']['seed'])
    pubmed_path = resolve_path(data_config['pubmed_path'])
    fake_path = resolve_path(data_config['fake_path'])
    output_dir = ensure_dir(data_config['processed_path'])
    if not pubmed_path.exists():
        raise FileNotFoundError(f'Real dataset not found: {pubmed_path}')
    if not fake_path.exists():
        raise FileNotFoundError(f'Fake dataset not found: {fake_path}')
    tokenizer = load_tokenizer_if_enabled(config)
    text_candidates = data_config.get('text_columns', ['abstract', 'text', 'content'])
    title_column = data_config.get('title_column', 'title')
    LOGGER.info('Loading raw data.')
    real_frame = standardize_frame(load_source(pubmed_path), text_candidates, label=0, title_column=title_column)
    fake_frame = standardize_frame(load_source(fake_path), text_candidates, label=1, title_column=title_column)
    merged = pd.concat([real_frame, fake_frame], ignore_index=True)
    merged = clean_frame(merged, min_words=data_config['min_words'], max_length=data_config['max_length'], tokenizer=tokenizer)
    LOGGER.info('Split cleaned dataset into train/val/test.')
    train_frame, val_frame, test_frame = split_frame(
        merged,
        train_ratio=data_config['train_split'],
        val_ratio=data_config['val_split'],
        test_ratio=data_config['test_split'],
        seed=config['project']['seed'],
    )
    split_paths = {
        'train': output_dir / data_config['train_file'],
        'val': output_dir / data_config['val_file'],
        'test': output_dir / data_config['test_file'],
    }
    train_frame.to_csv(split_paths['train'], index=False)
    val_frame.to_csv(split_paths['val'], index=False)
    test_frame.to_csv(split_paths['test'], index=False)
    summary = {
        'train_samples': len(train_frame),
        'val_samples': len(val_frame),
        'test_samples': len(test_frame),
        'output_dir': str(output_dir),
    }
    LOGGER.info('Prepared dataset: %s', summary)
    return summary


## Metrics And Visualization Utilities

These helpers reproduce the evaluation and plotting logic from the project. They are used both during training and during the notebook-only analysis section near the end.


In [ ]:
sns.set_theme(style='whitegrid', context='talk')

def compute_classification_metrics(labels: Sequence[int], probabilities: Sequence[float], threshold: float = 0.5) -> dict:
    label_array = np.asarray(labels)
    prob_array = np.asarray(probabilities, dtype=float)
    prob_array = np.clip(prob_array, 1e-7, 1.0 - 1e-7)
    predictions = (prob_array >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(label_array, predictions, labels=[0, 1]).ravel()
    specificity = float(tn / (tn + fp)) if (tn + fp) else 0.0
    metrics = {
        'accuracy': float(accuracy_score(label_array, predictions)),
        'precision': float(precision_score(label_array, predictions, zero_division=0)),
        'recall': float(recall_score(label_array, predictions, zero_division=0)),
        'specificity': specificity,
        'f1': float(f1_score(label_array, predictions, zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(label_array, predictions)),
        'mcc': float(matthews_corrcoef(label_array, predictions)) if len(np.unique(predictions)) > 1 else 0.0,
        'brier_score': float(brier_score_loss(label_array, prob_array)),
        'log_loss': float(log_loss(label_array, np.column_stack([1.0 - prob_array, prob_array]), labels=[0, 1])),
        'tp': int(tp),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'confidence_mean': float(np.mean(prob_array)),
        'confidence_std': float(np.std(prob_array)),
        'positive_prediction_rate': float(np.mean(predictions)),
    }
    if len(np.unique(label_array)) > 1:
        metrics['auc'] = float(roc_auc_score(label_array, prob_array))
        metrics['pr_auc'] = float(average_precision_score(label_array, prob_array))
    else:
        metrics['auc'] = math.nan
        metrics['pr_auc'] = math.nan
    return metrics

def compute_evasion_rate(detector_confidences: Sequence[float], evasion_threshold: float) -> float:
    if not detector_confidences:
        return 0.0
    evaded = sum(score < evasion_threshold for score in detector_confidences)
    return evaded / len(detector_confidences)

def compute_robustness_delta(current_auc: float, baseline_auc: float) -> float:
    if math.isnan(current_auc) or math.isnan(baseline_auc):
        return math.nan
    return current_auc - baseline_auc

def build_prediction_rows(texts: Sequence[str], labels: Sequence[int], probabilities: Sequence[float], threshold: float = 0.5) -> List[dict]:
    predictions = [int(score >= threshold) for score in probabilities]
    rows = []
    for text, label, probability, prediction in zip(texts, labels, probabilities, predictions):
        rows.append({'text': text, 'label': int(label), 'predicted_label': int(prediction), 'fake_probability': float(probability)})
    return rows

def compute_confidence_shift(before: Sequence[float], after: Sequence[float]) -> dict:
    if not before or not after:
        return {
            'mean_original_confidence': 0.0,
            'mean_rewritten_confidence': 0.0,
            'mean_confidence_drop': 0.0,
            'median_confidence_drop': 0.0,
        }
    before_array = np.asarray(before, dtype=float)
    after_array = np.asarray(after, dtype=float)
    drops = before_array - after_array
    return {
        'mean_original_confidence': float(before_array.mean()),
        'mean_rewritten_confidence': float(after_array.mean()),
        'mean_confidence_drop': float(drops.mean()),
        'median_confidence_drop': float(np.median(drops)),
    }

def _fallback_text_similarity(references: Sequence[str], candidates: Sequence[str]) -> float:
    scores: List[float] = []
    for reference, candidate in zip(references, candidates):
        reference_tokens = set(reference.lower().split())
        candidate_tokens = set(candidate.lower().split())
        if not reference_tokens and not candidate_tokens:
            scores.append(1.0)
            continue
        overlap = len(reference_tokens & candidate_tokens)
        denom = max(len(reference_tokens | candidate_tokens), 1)
        scores.append(overlap / denom)
    return float(np.mean(scores)) if scores else 0.0

def compute_rewrite_quality(references: Sequence[str], candidates: Sequence[str], model_type: str = 'microsoft/deberta-xlarge-mnli') -> float:
    if not references or not candidates:
        return 0.0
    try:
        from bert_score import score as bert_score
        _, _, f1 = bert_score(list(candidates), list(references), model_type=model_type, lang='en', verbose=False)
        return float(f1.mean().item())
    except Exception:
        return _fallback_text_similarity(references, candidates)

def average_metrics(history: Iterable[dict]) -> dict:
    history_list = list(history)
    if not history_list:
        return {}
    keys = sorted({key for row in history_list for key in row})
    summary = {}
    for key in keys:
        values = [row[key] for row in history_list if key in row and isinstance(row[key], (int, float))]
        summary[key] = float(np.mean(values)) if values else math.nan
    return summary

def plot_metric_history(history: Sequence[dict], output_path: str | Path, x_key: str, y_keys: Sequence[str], title: str, ylabel: str) -> Path:
    output_path = resolve_path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(10, 6))
    x_values = [row[x_key] for row in history if x_key in row]
    for key in y_keys:
        y_values = [row.get(key, math.nan) for row in history if x_key in row]
        plt.plot(x_values, y_values, marker='o', label=key)
    plt.title(title)
    plt.xlabel(x_key.replace('_', ' ').title())
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

def plot_round_metrics(history: Sequence[dict], output_dir: str | Path) -> Dict[str, Path]:
    output_dir = ensure_dir(output_dir)
    image_paths: Dict[str, Path] = {}
    metric_groups = {
        'round_performance': ['auc', 'f1', 'accuracy', 'balanced_accuracy'],
        'round_robustness': ['evasion_rate', 'rewrite_quality', 'robustness_delta'],
    }
    for name, keys in metric_groups.items():
        available = [key for key in keys if any(key in row for row in history)]
        if not available:
            continue
        image_paths[name] = plot_metric_history(history, output_dir / f'{name}.png', 'round', available, name.replace('_', ' ').title(), 'Metric Value')
    return image_paths

def plot_training_history(history: Sequence[dict], output_path: str | Path, title: str) -> Path:
    if not history:
        return resolve_path(output_path)
    numeric_keys = [key for key in history[0].keys() if key != 'epoch' and any(isinstance(row.get(key), (int, float)) for row in history)]
    return plot_metric_history(history, output_path, 'epoch', numeric_keys, title, 'Value')

def plot_confusion_heatmap(labels: Sequence[int], predictions: Sequence[int], output_path: str | Path, title: str = 'Confusion Matrix') -> Path:
    output_path = resolve_path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    plt.figure(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

def plot_roc(labels: Sequence[int], probabilities: Sequence[float], output_path: str | Path, title: str = 'ROC Curve') -> Path:
    output_path = resolve_path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fpr, tpr, _ = roc_curve(labels, probabilities)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label='ROC')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.title(title)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

def plot_precision_recall(labels: Sequence[int], probabilities: Sequence[float], output_path: str | Path, title: str = 'Precision-Recall Curve') -> Path:
    output_path = resolve_path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    precision, recall, _ = precision_recall_curve(labels, probabilities)
    plt.figure(figsize=(7, 6))
    plt.plot(recall, precision, label='PR')
    plt.title(title)
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

def plot_probability_histogram(labels: Sequence[int], probabilities: Sequence[float], output_path: str | Path, title: str = 'Prediction Confidence Distribution') -> Path:
    output_path = resolve_path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame({'label': labels, 'probability': probabilities})
    frame['label_name'] = frame['label'].map({0: 'Real', 1: 'Fake'})
    plt.figure(figsize=(8, 6))
    sns.histplot(data=frame, x='probability', hue='label_name', bins=20, kde=True, stat='density', common_norm=False)
    plt.title(title)
    plt.xlabel('Predicted Fake Probability')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()
    return output_path

def generate_classification_plots(labels: Sequence[int], probabilities: Sequence[float], predictions: Sequence[int], output_dir: str | Path, prefix: str) -> Dict[str, Path]:
    output_dir = ensure_dir(output_dir)
    paths = {
        f'{prefix}_confusion_matrix': plot_confusion_heatmap(labels, predictions, output_dir / f'{prefix}_confusion_matrix.png'),
        f'{prefix}_confidence_histogram': plot_probability_histogram(labels, probabilities, output_dir / f'{prefix}_confidence_histogram.png'),
    }
    if len(set(labels)) > 1:
        paths[f'{prefix}_roc_curve'] = plot_roc(labels, probabilities, output_dir / f'{prefix}_roc_curve.png')
        paths[f'{prefix}_pr_curve'] = plot_precision_recall(labels, probabilities, output_dir / f'{prefix}_pr_curve.png')
    return paths


## BioBERT Detector Implementation

This cell embeds the detector dataset classes, training loop, evaluation helper, and prediction helper directly into the notebook. The logic is aligned with the detector module from the original project.


In [ ]:
class SequenceClassificationDataset(Dataset):
    def __init__(self, texts: Sequence[str], labels: Optional[Sequence[int]], tokenizer: PreTrainedTokenizerBase, max_length: int) -> None:
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, index: int) -> dict:
        encoding = self.tokenizer(self.texts[index], truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        item = {key: value.squeeze(0) for key, value in encoding.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

def read_split(path: str | Path) -> pd.DataFrame:
    frame = pd.read_csv(resolve_path(path))
    if 'text' not in frame.columns or 'label' not in frame.columns:
        raise ValueError(f'{path} must contain "text" and "label" columns.')
    return frame

def create_dataloader(frame: pd.DataFrame, tokenizer: PreTrainedTokenizerBase, max_length: int, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = SequenceClassificationDataset(frame['text'].astype(str).tolist(), frame['label'].astype(int).tolist(), tokenizer, max_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def create_inference_dataloader(texts: Sequence[str], tokenizer: PreTrainedTokenizerBase, max_length: int, batch_size: int) -> DataLoader:
    dataset = SequenceClassificationDataset(list(texts), None, tokenizer, max_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=False)

def maybe_enable_gradient_checkpointing(model: PreTrainedModel, enabled: bool) -> None:
    if enabled and hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
        model.config.use_cache = False

@dataclass
class DetectorPredictor:
    model: PreTrainedModel
    tokenizer: PreTrainedTokenizerBase
    device: torch.device
    max_length: int
    batch_size: int

    @torch.inference_mode()
    def score_texts(self, texts: Sequence[str]) -> List[float]:
        if not texts:
            return []
        loader = create_inference_dataloader(texts, self.tokenizer, self.max_length, self.batch_size)
        self.model.eval()
        scores: List[float] = []
        for batch in loader:
            batch = {key: value.to(self.device) for key, value in batch.items()}
            outputs = self.model(**batch)
            probabilities = torch.softmax(outputs.logits, dim=-1)[:, 1]
            scores.extend(probabilities.detach().cpu().tolist())
        return scores

def load_predictor(model_dir: str | Path, max_length: int, batch_size: int, device_name: str = 'auto') -> DetectorPredictor:
    device = get_device(device_name)
    resolved_dir = resolve_path(model_dir)
    tokenizer = AutoTokenizer.from_pretrained(resolved_dir)
    model = AutoModelForSequenceClassification.from_pretrained(resolved_dir)
    model.to(device)
    model.eval()
    return DetectorPredictor(model=model, tokenizer=tokenizer, device=device, max_length=max_length, batch_size=batch_size)

def evaluate_model(model: PreTrainedModel, dataloader: DataLoader, device: torch.device, threshold: float = 0.5, return_predictions: bool = False) -> dict | tuple[dict, List[int], List[float]]:
    model.eval()
    losses: List[float] = []
    labels: List[int] = []
    probabilities: List[float] = []
    with torch.inference_mode():
        for batch in dataloader:
            labels.extend(batch['labels'].tolist())
            batch = {key: value.to(device) for key, value in batch.items()}
            outputs = model(**batch)
            losses.append(outputs.loss.detach().cpu().item())
            scores = torch.softmax(outputs.logits, dim=-1)[:, 1]
            probabilities.extend(scores.detach().cpu().tolist())
    metrics = compute_classification_metrics(labels, probabilities, threshold=threshold)
    metrics['loss'] = float(sum(losses) / max(len(losses), 1))
    if return_predictions:
        return metrics, labels, probabilities
    return metrics

def train_detector(config: dict, train_path: Optional[str | Path] = None, val_path: Optional[str | Path] = None, output_dir: Optional[str | Path] = None, init_checkpoint: Optional[str | Path] = None) -> dict:
    detector_config = config['detector']
    data_config = config['data']
    project_config = config['project']
    set_seed(project_config['seed'])
    device = get_device(project_config.get('device', 'auto'))

    train_path = resolve_path(train_path or Path(data_config['processed_path']) / data_config['train_file'])
    val_path = resolve_path(val_path or Path(data_config['processed_path']) / data_config['val_file'])
    output_dir = ensure_dir(output_dir or Path(detector_config['checkpoint_dir']) / 'baseline')

    train_frame = read_split(train_path)
    val_frame = read_split(val_path)

    model_source = str(resolve_path(init_checkpoint)) if init_checkpoint else detector_config['model_name']
    tokenizer = AutoTokenizer.from_pretrained(model_source)
    model = AutoModelForSequenceClassification.from_pretrained(model_source, num_labels=detector_config['num_labels'])
    maybe_enable_gradient_checkpointing(model, detector_config.get('gradient_checkpointing', False))
    model.to(device)

    train_loader = create_dataloader(train_frame, tokenizer, detector_config['max_length'], detector_config['batch_size'], True)
    val_loader = create_dataloader(val_frame, tokenizer, detector_config['max_length'], detector_config['batch_size'], False)

    optimizer = AdamW(model.parameters(), lr=detector_config['lr'], weight_decay=detector_config['weight_decay'])
    total_steps = math.ceil(len(train_loader) / detector_config['grad_accum_steps']) * detector_config['epochs_per_round']
    warmup_steps = int(total_steps * detector_config['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=warmup_steps, num_training_steps=max(total_steps, 1))

    use_amp = detector_config.get('fp16', False) and device.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    logger = ExperimentLogger(config=config, run_name=f'detector_{Path(output_dir).name}', job_type='detector_train', tags=['detector', 'biobert'])

    best_metrics: Optional[dict] = None
    best_score = float('-inf')
    global_step = 0
    epoch_history: List[dict] = []
    best_prediction_payload: Optional[dict] = None

    LOGGER.info('Training detector on %s using %s', train_path, device)
    for epoch in range(detector_config['epochs_per_round']):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        progress = tqdm(train_loader, desc=f'detector epoch {epoch + 1}', leave=False)
        train_losses: List[float] = []
        for step, batch in enumerate(progress, start=1):
            batch = {key: value.to(device) for key, value in batch.items()}
            with maybe_autocast(use_amp, device):
                outputs = model(**batch)
                loss = outputs.loss / detector_config['grad_accum_steps']
            train_losses.append(loss.item() * detector_config['grad_accum_steps'])
            scaler.scale(loss).backward()
            if step % detector_config['grad_accum_steps'] == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
            progress.set_postfix(loss=f'{loss.item() * detector_config["grad_accum_steps"]:.4f}')

        val_metrics, val_labels, val_probabilities = evaluate_model(model, val_loader, device, threshold=config['evaluation'].get('threshold', 0.5), return_predictions=True)
        epoch_metrics = {'epoch': epoch + 1, 'train_loss': float(sum(train_losses) / max(len(train_losses), 1)), **val_metrics}
        epoch_history.append(epoch_metrics)
        LOGGER.info('Detector epoch %s validation metrics: %s', epoch + 1, val_metrics)
        current_score = val_metrics.get('f1', 0.0)
        logger.log_metrics(epoch_metrics, step=epoch + 1, prefix='detector')

        if current_score >= best_score:
            best_score = current_score
            best_metrics = val_metrics
            best_prediction_payload = {'labels': val_labels, 'probabilities': val_probabilities}
            model.save_pretrained(output_dir)
            tokenizer.save_pretrained(output_dir)
            save_json({'metrics': best_metrics, 'train_path': str(train_path), 'val_path': str(val_path), 'global_step': global_step, 'epoch_history': epoch_history}, Path(output_dir) / 'training_summary.json')

    image_paths = {'detector_training_curve': plot_training_history(epoch_history, Path(output_dir) / 'detector_training_curve.png', title='Detector Training History')}
    if best_prediction_payload:
        prediction_frame = pd.DataFrame(build_prediction_rows(val_frame['text'].astype(str).tolist(), best_prediction_payload['labels'], best_prediction_payload['probabilities'], threshold=config['evaluation'].get('threshold', 0.5)))
        prediction_frame.to_csv(Path(output_dir) / 'val_predictions.csv', index=False)
        image_paths.update(generate_classification_plots(best_prediction_payload['labels'], best_prediction_payload['probabilities'], prediction_frame['predicted_label'].astype(int).tolist(), Path(output_dir), 'val'))
        logger.log_dataframe('detector_val_predictions', prediction_frame)

    logger.log_images_from_paths(image_paths)
    summary = {'output_dir': str(output_dir), 'best_metrics': best_metrics or {}, 'steps': global_step, 'epoch_history': epoch_history}
    logger.log_metrics(summary['best_metrics'], prefix='detector_best')
    logger.finish()
    LOGGER.info('Detector training complete: %s', summary)
    return summary

def score_texts(model_dir: str | Path, texts: Sequence[str], config: dict) -> List[float]:
    predictor = load_predictor(model_dir=model_dir, max_length=config['detector']['max_length'], batch_size=config['detector']['batch_size'], device_name=config['project'].get('device', 'auto'))
    return predictor.score_texts(texts)


## SeqGAN Implementation

This cell brings the SeqGAN pieces into the notebook: vocabulary handling, generator, discriminator, rollout policy, pretraining, adversarial updates, checkpoint save/load, and fake-text generation.


In [ ]:
PAD_TOKEN = '<pad>'
BOS_TOKEN = '<bos>'
EOS_TOKEN = '<eos>'
UNK_TOKEN = '<unk>'
SPECIAL_TOKENS = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

@dataclass
class Vocabulary:
    stoi: Dict[str, int]
    itos: Dict[int, str]

    @property
    def pad_token_id(self) -> int:
        return self.stoi[PAD_TOKEN]

    @property
    def bos_token_id(self) -> int:
        return self.stoi[BOS_TOKEN]

    @property
    def eos_token_id(self) -> int:
        return self.stoi[EOS_TOKEN]

    @property
    def unk_token_id(self) -> int:
        return self.stoi[UNK_TOKEN]

    def encode(self, text: str, seq_len: int) -> List[int]:
        tokens = [BOS_TOKEN] + text.lower().split()[: seq_len - 2] + [EOS_TOKEN]
        token_ids = [self.stoi.get(token, self.unk_token_id) for token in tokens]
        if len(token_ids) < seq_len:
            token_ids += [self.pad_token_id] * (seq_len - len(token_ids))
        return token_ids[:seq_len]

    def decode(self, token_ids: Sequence[int]) -> str:
        tokens: List[str] = []
        for token_id in token_ids:
            token = self.itos.get(int(token_id), UNK_TOKEN)
            if token in {PAD_TOKEN, BOS_TOKEN}:
                continue
            if token == EOS_TOKEN:
                break
            tokens.append(token)
        return ' '.join(tokens)

    def to_dict(self) -> dict:
        return {'stoi': self.stoi, 'itos': {str(key): value for key, value in self.itos.items()}}

    @classmethod
    def from_dict(cls, payload: dict) -> 'Vocabulary':
        return cls(stoi={key: int(value) for key, value in payload['stoi'].items()}, itos={int(key): value for key, value in payload['itos'].items()})

@dataclass
class GeneratorConfig:
    vocab_size: int
    embed_dim: int
    hidden_dim: int
    num_layers: int
    pad_token_id: int = 0

class SeqGANGenerator(nn.Module):
    def __init__(self, config: GeneratorConfig) -> None:
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.embed_dim, padding_idx=config.pad_token_id)
        self.lstm = nn.LSTM(config.embed_dim, config.hidden_dim, config.num_layers, batch_first=True)
        self.output = nn.Linear(config.hidden_dim, config.vocab_size)

    def forward(self, input_ids: torch.Tensor, hidden: Optional[Tuple[torch.Tensor, torch.Tensor]] = None) -> tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        embeddings = self.embedding(input_ids)
        states, hidden = self.lstm(embeddings, hidden)
        logits = self.output(states)
        return logits, hidden

    def pretrain_loss(self, input_ids: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
        logits, _ = self.forward(input_ids)
        return nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), target_ids.reshape(-1), ignore_index=self.config.pad_token_id)

    def policy_gradient_loss(self, sampled_ids: torch.Tensor, rewards: torch.Tensor) -> torch.Tensor:
        logits, _ = self.forward(sampled_ids[:, :-1])
        log_probs = nn.functional.log_softmax(logits, dim=-1)
        chosen_log_probs = torch.gather(log_probs, dim=-1, index=sampled_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        loss = -(chosen_log_probs * rewards).mean()
        return loss

    @torch.inference_mode()
    def sample(self, batch_size: int, seq_len: int, bos_token_id: int, eos_token_id: Optional[int] = None, temperature: float = 1.0, device: Optional[torch.device] = None) -> torch.Tensor:
        device = device or next(self.parameters()).device
        generated = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)
        hidden = None
        for _ in range(seq_len - 1):
            logits, hidden = self.forward(generated[:, -1:].to(device), hidden)
            step_logits = logits[:, -1, :] / max(temperature, 1e-5)
            distribution = Categorical(logits=step_logits)
            next_tokens = distribution.sample().unsqueeze(-1)
            generated = torch.cat([generated, next_tokens], dim=1)
            if eos_token_id is not None and torch.all(next_tokens.squeeze(-1) == eos_token_id):
                break
        return generated

    @torch.inference_mode()
    def complete_sequences(self, prefixes: torch.Tensor, target_len: int, temperature: float = 1.0) -> torch.Tensor:
        device = prefixes.device
        generated = prefixes.clone()
        hidden = None
        if prefixes.size(1) > 1:
            _, hidden = self.forward(prefixes[:, :-1])
        current = prefixes[:, -1:]
        while generated.size(1) < target_len:
            logits, hidden = self.forward(current, hidden)
            step_logits = logits[:, -1, :] / max(temperature, 1e-5)
            distribution = Categorical(logits=step_logits)
            current = distribution.sample().unsqueeze(-1)
            generated = torch.cat([generated, current.to(device)], dim=1)
        return generated

@dataclass
class DiscriminatorConfig:
    vocab_size: int
    embed_dim: int
    hidden_dim: int
    num_layers: int
    pad_token_id: int = 0

class SeqGANDiscriminator(nn.Module):
    def __init__(self, config: DiscriminatorConfig) -> None:
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(config.vocab_size, config.embed_dim, padding_idx=config.pad_token_id)
        self.lstm = nn.LSTM(config.embed_dim, config.hidden_dim, config.num_layers, batch_first=True)
        self.classifier = nn.Linear(config.hidden_dim, 1)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        states, _ = self.lstm(embedded)
        pooled = states[:, -1, :]
        logits = self.classifier(pooled).squeeze(-1)
        return logits

    def loss(self, input_ids: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        logits = self.forward(input_ids)
        return nn.functional.binary_cross_entropy_with_logits(logits, labels.float())

    @torch.inference_mode()
    def predict_proba(self, input_ids: torch.Tensor) -> torch.Tensor:
        logits = self.forward(input_ids)
        return torch.sigmoid(logits)

class RolloutPolicy:
    def __init__(self, generator: SeqGANGenerator, update_rate: float = 0.8) -> None:
        self.generator = copy.deepcopy(generator)
        self.update_rate = update_rate

    def update_params(self, source_generator: SeqGANGenerator) -> None:
        for target_param, source_param in zip(self.generator.parameters(), source_generator.parameters()):
            target_param.data.mul_(self.update_rate).add_(source_param.data * (1.0 - self.update_rate))

    @torch.inference_mode()
    def get_rewards(self, sampled_ids: torch.Tensor, rollout_num: int, discriminator: SeqGANDiscriminator, temperature: float = 1.0) -> torch.Tensor:
        batch_size, seq_len = sampled_ids.size()
        device = sampled_ids.device
        rewards = torch.zeros(batch_size, seq_len - 1, device=device)
        for _ in range(rollout_num):
            for prefix_length in range(1, seq_len):
                prefixes = sampled_ids[:, :prefix_length]
                completed = self.generator.complete_sequences(prefixes=prefixes, target_len=seq_len, temperature=temperature)
                scores = discriminator.predict_proba(completed)
                rewards[:, prefix_length - 1] += scores
        rewards /= max(rollout_num, 1)
        return rewards

class GeneratorPretrainDataset(Dataset):
    def __init__(self, sequences: Sequence[Sequence[int]]) -> None:
        self.sequences = [torch.tensor(sequence, dtype=torch.long) for sequence in sequences]

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> dict:
        sequence = self.sequences[index]
        return {'input_ids': sequence[:-1], 'target_ids': sequence[1:]}

class SequenceDataset(Dataset):
    def __init__(self, sequences: Sequence[Sequence[int]], labels: Sequence[int]) -> None:
        self.sequences = [torch.tensor(sequence, dtype=torch.long) for sequence in sequences]
        self.labels = [torch.tensor(label, dtype=torch.float32) for label in labels]

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.sequences[index], self.labels[index]

def build_vocab(texts: Sequence[str], vocab_size: int) -> Vocabulary:
    counts: Dict[str, int] = {}
    for text in texts:
        for token in text.lower().split():
            counts[token] = counts.get(token, 0) + 1
    most_common = sorted(counts.items(), key=lambda item: item[1], reverse=True)[: max(vocab_size - len(SPECIAL_TOKENS), 0)]
    stoi = {token: index for index, token in enumerate(SPECIAL_TOKENS)}
    for token, _ in most_common:
        if token not in stoi:
            stoi[token] = len(stoi)
    itos = {index: token for token, index in stoi.items()}
    return Vocabulary(stoi=stoi, itos=itos)

def load_training_texts(train_path: str | Path) -> List[str]:
    frame = pd.read_csv(resolve_path(train_path))
    if 'text' not in frame.columns:
        raise ValueError(f'{train_path} must contain a "text" column.')
    return frame['text'].dropna().astype(str).tolist()

def prepare_sequences(texts: Sequence[str], vocab: Vocabulary, seq_len: int) -> List[List[int]]:
    return [vocab.encode(text, seq_len) for text in texts]

def pretrain_generator(generator: SeqGANGenerator, dataloader: DataLoader, optimizer: Adam, epochs: int, device: torch.device, gradient_clip_norm: float, logger: ExperimentLogger | None = None) -> List[dict]:
    history: List[dict] = []
    generator.train()
    for epoch in range(epochs):
        losses: List[float] = []
        for batch in tqdm(dataloader, desc=f'seqgan generator pretrain {epoch + 1}', leave=False):
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['target_ids'].to(device)
            loss = generator.pretrain_loss(input_ids, target_ids)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), gradient_clip_norm)
            optimizer.step()
            losses.append(loss.item())
        epoch_metrics = {'epoch': epoch + 1, 'generator_pretrain_loss': float(sum(losses) / max(len(losses), 1))}
        history.append(epoch_metrics)
        if logger:
            logger.log_metrics(epoch_metrics, step=epoch + 1, prefix='seqgan_generator_pretrain')
    return history

def pretrain_discriminator(discriminator: SeqGANDiscriminator, generator: SeqGANGenerator, real_sequences: Sequence[Sequence[int]], batch_size: int, epochs: int, device: torch.device, bos_token_id: int, seq_len: int, lr: float, logger: ExperimentLogger | None = None) -> List[dict]:
    optimizer = Adam(discriminator.parameters(), lr=lr)
    history: List[dict] = []
    sampled_real_count = min(len(real_sequences), batch_size * 20)
    for epoch in range(epochs):
        sampled_reals = random.sample(list(real_sequences), k=sampled_real_count)
        fake_sequences = generator.sample(batch_size=sampled_real_count, seq_len=seq_len, bos_token_id=bos_token_id, device=device).cpu().tolist()
        sequences = sampled_reals + fake_sequences
        labels = [1] * len(sampled_reals) + [0] * len(fake_sequences)
        dataset = SequenceDataset(sequences, labels)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        losses: List[float] = []
        discriminator.train()
        for input_ids, label_batch in tqdm(dataloader, desc=f'seqgan discriminator pretrain {epoch + 1}', leave=False):
            input_ids = input_ids.to(device)
            label_batch = label_batch.to(device)
            loss = discriminator.loss(input_ids, label_batch)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        epoch_metrics = {'epoch': epoch + 1, 'discriminator_pretrain_loss': float(sum(losses) / max(len(losses), 1))}
        history.append(epoch_metrics)
        if logger:
            logger.log_metrics(epoch_metrics, step=epoch + 1, prefix='seqgan_discriminator_pretrain')
    return history

def adversarial_train(generator: SeqGANGenerator, discriminator: SeqGANDiscriminator, rollout: RolloutPolicy, config: dict, device: torch.device, bos_token_id: int, real_sequences: Sequence[Sequence[int]], logger: ExperimentLogger | None = None) -> List[dict]:
    seqgan_config = config['seqgan']
    g_optimizer = Adam(generator.parameters(), lr=seqgan_config['lr_g'])
    d_optimizer = Adam(discriminator.parameters(), lr=seqgan_config['lr_d'])
    history: List[dict] = []
    for epoch in range(seqgan_config['adversarial_epochs']):
        generator.train()
        sampled_ids = generator.sample(batch_size=seqgan_config['batch_size'], seq_len=seqgan_config['seq_len'], bos_token_id=bos_token_id, device=device, temperature=seqgan_config['temperature'])
        rewards = rollout.get_rewards(sampled_ids=sampled_ids, rollout_num=seqgan_config['rollout_num'], discriminator=discriminator, temperature=seqgan_config['temperature'])
        g_loss = generator.policy_gradient_loss(sampled_ids, rewards)
        g_optimizer.zero_grad(set_to_none=True)
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), seqgan_config['gradient_clip_norm'])
        g_optimizer.step()

        discriminator.train()
        fake_batch = sampled_ids.detach()
        sampled_reals = random.sample(list(real_sequences), k=min(len(real_sequences), fake_batch.size(0)))
        real_batch = torch.tensor(sampled_reals, dtype=torch.long, device=device)
        min_batch = min(real_batch.size(0), fake_batch.size(0))
        mixed_input = torch.cat([real_batch[:min_batch], fake_batch[:min_batch]], dim=0)
        labels = torch.cat([torch.ones(min_batch, device=device), torch.zeros(min_batch, device=device)], dim=0)
        d_loss = discriminator.loss(mixed_input, labels)
        d_optimizer.zero_grad(set_to_none=True)
        d_loss.backward()
        d_optimizer.step()

        rollout.update_params(generator)
        epoch_metrics = {'epoch': epoch + 1, 'generator_adversarial_loss': float(g_loss.detach().cpu().item()), 'discriminator_adversarial_loss': float(d_loss.detach().cpu().item())}
        history.append(epoch_metrics)
        if logger:
            logger.log_metrics(epoch_metrics, step=epoch + 1, prefix='seqgan_adversarial')
    return history

def compute_bleu_like_score(references: Sequence[str], candidates: Sequence[str]) -> float:
    if not references or not candidates:
        return 0.0
    scores: List[float] = []
    for reference, candidate in zip(references, candidates):
        reference_tokens = reference.split()
        candidate_tokens = candidate.split()
        if not candidate_tokens:
            scores.append(0.0)
            continue
        overlap = sum(1 for token in candidate_tokens if token in reference_tokens)
        scores.append(overlap / len(candidate_tokens))
    return float(sum(scores) / len(scores))

def compute_perplexity_like(generator_losses: Sequence[float]) -> float:
    if not generator_losses:
        return math.nan
    return float(math.exp(sum(generator_losses) / len(generator_losses)))

def save_seqgan_checkpoint(output_dir: Path, generator: SeqGANGenerator, discriminator: SeqGANDiscriminator, vocab: Vocabulary, config: dict, metrics: dict) -> None:
    output_dir = ensure_dir(output_dir)
    torch.save(
        {
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'generator_config': asdict(generator.config),
            'discriminator_config': asdict(discriminator.config),
            'vocab': vocab.to_dict(),
            'metrics': metrics,
            'seqgan_config': config['seqgan'],
        },
        output_dir / 'seqgan.pt',
    )
    save_json(metrics, output_dir / 'training_summary.json')

def load_seqgan_bundle(checkpoint_dir: str | Path, device_name: str = 'auto') -> dict:
    device = get_device(device_name)
    checkpoint_path = resolve_path(checkpoint_dir) / 'seqgan.pt'
    payload = torch.load(checkpoint_path, map_location=device)
    generator_config = GeneratorConfig(**payload['generator_config'])
    discriminator_config = DiscriminatorConfig(**payload['discriminator_config'])
    vocab = Vocabulary.from_dict(payload['vocab'])
    generator = SeqGANGenerator(generator_config)
    discriminator = SeqGANDiscriminator(discriminator_config)
    generator.load_state_dict(payload['generator_state_dict'])
    discriminator.load_state_dict(payload['discriminator_state_dict'])
    generator.to(device).eval()
    discriminator.to(device).eval()
    return {'generator': generator, 'discriminator': discriminator, 'vocab': vocab, 'config': payload['seqgan_config'], 'metrics': payload.get('metrics', {}), 'device': device}

@torch.inference_mode()
def generate_fake_texts(checkpoint_dir: str | Path, num_samples: int, device_name: str = 'auto') -> List[str]:
    bundle = load_seqgan_bundle(checkpoint_dir, device_name=device_name)
    generator: SeqGANGenerator = bundle['generator']
    vocab: Vocabulary = bundle['vocab']
    seqgan_config: dict = bundle['config']
    device: torch.device = bundle['device']
    sampled = generator.sample(batch_size=num_samples, seq_len=seqgan_config['seq_len'], bos_token_id=vocab.bos_token_id, eos_token_id=vocab.eos_token_id, temperature=seqgan_config.get('temperature', 1.0), device=device)
    return [vocab.decode(row.tolist()) for row in sampled]

def train_seqgan(config: dict, train_path: str | Path | None = None, output_dir: str | Path | None = None) -> dict:
    set_seed(config['project']['seed'])
    seqgan_config = config['seqgan']
    device = get_device(config['project'].get('device', 'auto'))
    logger = ExperimentLogger(config=config, run_name=f'seqgan_{Path(output_dir or seqgan_config["checkpoint_dir"]).name}', job_type='seqgan_train', tags=['seqgan'])

    train_path = resolve_path(train_path or Path(config['data']['processed_path']) / config['data']['train_file'])
    output_dir = ensure_dir(output_dir or seqgan_config['checkpoint_dir'])

    texts = load_training_texts(train_path)
    vocab = build_vocab(texts, vocab_size=seqgan_config['vocab_size'])
    sequences = prepare_sequences(texts, vocab=vocab, seq_len=seqgan_config['seq_len'])

    generator = SeqGANGenerator(GeneratorConfig(vocab_size=len(vocab.stoi), embed_dim=seqgan_config['embed_dim'], hidden_dim=seqgan_config['hidden_dim'], num_layers=seqgan_config['num_layers'], pad_token_id=vocab.pad_token_id)).to(device)
    discriminator = SeqGANDiscriminator(DiscriminatorConfig(vocab_size=len(vocab.stoi), embed_dim=seqgan_config['embed_dim'], hidden_dim=seqgan_config['hidden_dim'], num_layers=seqgan_config['num_layers'], pad_token_id=vocab.pad_token_id)).to(device)

    generator_dataset = GeneratorPretrainDataset(sequences)
    generator_loader = DataLoader(generator_dataset, batch_size=seqgan_config['batch_size'], shuffle=True)
    generator_optimizer = Adam(generator.parameters(), lr=seqgan_config['lr_g'])
    generator_history = pretrain_generator(generator, generator_loader, generator_optimizer, seqgan_config['pretrain_epochs_g'], device, seqgan_config['gradient_clip_norm'], logger)

    discriminator_history = pretrain_discriminator(discriminator, generator, sequences, seqgan_config['batch_size'], seqgan_config['pretrain_epochs_d'], device, vocab.bos_token_id, seqgan_config['seq_len'], seqgan_config['lr_d'], logger)

    rollout = RolloutPolicy(generator)
    adversarial_history = adversarial_train(generator, discriminator, rollout, config, device, vocab.bos_token_id, sequences, logger)

    generated_texts = [vocab.decode(row.tolist()) for row in generator.sample(batch_size=min(32, len(texts)), seq_len=seqgan_config['seq_len'], bos_token_id=vocab.bos_token_id, eos_token_id=vocab.eos_token_id, device=device)]
    reference_texts = texts[: len(generated_texts)]
    metrics = {
        'generator': average_metrics(generator_history),
        'discriminator': average_metrics(discriminator_history),
        'adversarial': average_metrics(adversarial_history),
        'bleu_like': compute_bleu_like_score(reference_texts, generated_texts),
        'perplexity_like': compute_perplexity_like([row['generator_pretrain_loss'] for row in generator_history if 'generator_pretrain_loss' in row]),
    }

    save_seqgan_checkpoint(Path(output_dir), generator, discriminator, vocab, config, metrics)
    image_paths = {
        'seqgan_generator_pretrain_curve': plot_training_history(generator_history, Path(output_dir) / 'seqgan_generator_pretrain_curve.png', title='SeqGAN Generator Pretrain'),
        'seqgan_discriminator_pretrain_curve': plot_training_history(discriminator_history, Path(output_dir) / 'seqgan_discriminator_pretrain_curve.png', title='SeqGAN Discriminator Pretrain'),
        'seqgan_adversarial_curve': plot_training_history(adversarial_history, Path(output_dir) / 'seqgan_adversarial_curve.png', title='SeqGAN Adversarial Training'),
    }
    logger.log_metrics(metrics['generator'], prefix='seqgan_generator_summary')
    logger.log_metrics(metrics['discriminator'], prefix='seqgan_discriminator_summary')
    logger.log_metrics(metrics['adversarial'], prefix='seqgan_adversarial_summary')
    logger.log_metrics({'bleu_like': metrics['bleu_like'], 'perplexity_like': metrics['perplexity_like']}, prefix='seqgan_eval')
    logger.log_images_from_paths(image_paths)
    logger.finish()
    LOGGER.info('SeqGAN training complete: %s', metrics)
    return {'output_dir': str(output_dir), 'metrics': metrics}


## BioGPT Adversarial Agent Implementation

This cell embeds the adversarial agent logic used by the project. It supports rewrite generation, free fake-text generation, and LoRA fine-tuning from successful evasions. Like the original project, the fine-tuning path is optional and depends on `peft` being available.


In [ ]:
REWRITE_TEMPLATE = """System: You are a biomedical text writer. Rewrite the following abstract
to sound like a credible scientific publication while preserving its
core claims. Do not add new factual information.

Original: {fake_abstract}

Rewritten:
"""

GENERATE_TEMPLATE = """System: Write a convincing but fictitious biomedical abstract about
{topic}. It should follow standard scientific writing conventions
and sound plausible to a domain expert.

Abstract:
"""

def build_rewrite_prompt(fake_abstract: str) -> str:
    return REWRITE_TEMPLATE.format(fake_abstract=fake_abstract.strip())

def build_generation_prompt(topic: str) -> str:
    return GENERATE_TEMPLATE.format(topic=topic.strip())

try:
    from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
except Exception:
    LoraConfig = None
    PeftModel = None
    TaskType = None
    get_peft_model = None
    prepare_model_for_kbit_training = None

class PromptResponseDataset(Dataset):
    def __init__(self, tokenizer, examples: Sequence[tuple[str, str]], max_length: int) -> None:
        self.records = []
        self.tokenizer = tokenizer
        for prompt, response in examples:
            prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
            response_ids = tokenizer(response, add_special_tokens=False)['input_ids']
            input_ids = prompt_ids + response_ids + [tokenizer.eos_token_id]
            input_ids = input_ids[:max_length]
            attention_mask = [1] * len(input_ids)
            labels = [-100] * min(len(prompt_ids), len(input_ids))
            remaining_response = input_ids[len(labels):]
            labels.extend(remaining_response)
            pad_length = max_length - len(input_ids)
            if pad_length > 0:
                input_ids += [tokenizer.pad_token_id] * pad_length
                attention_mask += [0] * pad_length
                labels += [-100] * pad_length
            self.records.append({'input_ids': torch.tensor(input_ids, dtype=torch.long), 'attention_mask': torch.tensor(attention_mask, dtype=torch.long), 'labels': torch.tensor(labels, dtype=torch.long)})

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> dict:
        return self.records[index]

@dataclass
class SuccessfulEvasionExample:
    original_text: str
    adversarial_text: str
    detector_confidence: float

def _load_quantization_config(use_4bit: bool) -> Optional['BitsAndBytesConfig']:
    if not use_4bit or not torch.cuda.is_available():
        return None
    try:
        from transformers import BitsAndBytesConfig
        return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    except Exception as exc:
        LOGGER.warning('4-bit loading requested but unavailable, falling back to standard weights: %s', exc)
        return None

class AdversarialAgent:
    def __init__(self, config: dict, checkpoint_dir: Optional[str | Path] = None) -> None:
        self.config = config
        self.agent_config = config['agent']
        self.device = get_device(config['project'].get('device', 'auto'))
        self.tokenizer = AutoTokenizer.from_pretrained(self.agent_config['model_name'])
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        quantization_config = _load_quantization_config(self.agent_config.get('use_4bit', False))
        model_kwargs = {}
        if quantization_config is not None:
            model_kwargs['quantization_config'] = quantization_config
            model_kwargs['device_map'] = 'auto'
        self.model = AutoModelForCausalLM.from_pretrained(self.agent_config['model_name'], **model_kwargs)
        if checkpoint_dir and resolve_path(checkpoint_dir).exists() and PeftModel is not None:
            try:
                self.model = PeftModel.from_pretrained(self.model, resolve_path(checkpoint_dir))
            except Exception:
                pass
        if self.device.type == 'cuda' and 'device_map' not in model_kwargs:
            self.model.to(self.device)
        self.model.eval()

    def _generate(self, prompts: Sequence[str], max_new_tokens: Optional[int] = None) -> List[str]:
        if not prompts:
            return []
        encoded = self.tokenizer(list(prompts), padding=True, truncation=True, max_length=self.agent_config['max_length'], return_tensors='pt')
        encoded = {key: value.to(self.model.device) for key, value in encoded.items()}
        with torch.inference_mode():
            outputs = self.model.generate(
                **encoded,
                do_sample=True,
                temperature=self.agent_config['temperature'],
                top_p=self.agent_config['top_p'],
                max_new_tokens=max_new_tokens or self.agent_config['max_new_tokens'],
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        generated: List[str] = []
        prompt_lengths = encoded['attention_mask'].sum(dim=1).tolist()
        for row_index, generated_ids in enumerate(outputs):
            new_token_ids = generated_ids[int(prompt_lengths[row_index]):]
            generated.append(self.tokenizer.decode(new_token_ids, skip_special_tokens=True).strip())
        return generated

    def rewrite(self, texts: Sequence[str]) -> List[str]:
        prompts = [build_rewrite_prompt(text) for text in texts]
        return self._generate(prompts)

    def generate(self, topics: Sequence[str]) -> List[str]:
        prompts = [build_generation_prompt(topic) for topic in topics]
        return self._generate(prompts)

    def collect_successful_examples(self, originals: Sequence[str], adversarial_texts: Sequence[str], detector_confidences: Sequence[float]) -> List[SuccessfulEvasionExample]:
        examples: List[SuccessfulEvasionExample] = []
        for original, rewritten, confidence in zip(originals, adversarial_texts, detector_confidences):
            if confidence < self.agent_config['evasion_threshold']:
                examples.append(SuccessfulEvasionExample(original_text=original, adversarial_text=rewritten, detector_confidence=confidence))
        return examples

    def finetune(self, examples: Sequence[SuccessfulEvasionExample], output_dir: str | Path) -> dict:
        if not examples:
            LOGGER.info('No successful evasion examples collected; skipping agent fine-tuning.')
            return {'output_dir': str(output_dir), 'trainable_parameters': 0, 'examples': 0}
        if LoraConfig is None or get_peft_model is None:
            raise ImportError('peft is required to fine-tune the adversarial agent.')

        output_dir = ensure_dir(output_dir)
        set_seed(self.config['project']['seed'])
        quantization_config = _load_quantization_config(self.agent_config.get('use_4bit', False))
        model_kwargs = {}
        if quantization_config is not None:
            model_kwargs['quantization_config'] = quantization_config
            model_kwargs['device_map'] = 'auto'

        model = AutoModelForCausalLM.from_pretrained(self.agent_config['model_name'], **model_kwargs)
        if quantization_config is not None and prepare_model_for_kbit_training is not None:
            model = prepare_model_for_kbit_training(model)
        elif self.device.type == 'cuda' and 'device_map' not in model_kwargs:
            model.to(self.device)

        if self.agent_config.get('gradient_checkpointing', False) and hasattr(model, 'gradient_checkpointing_enable'):
            model.gradient_checkpointing_enable()
            model.config.use_cache = False

        lora_config = LoraConfig(r=self.agent_config['lora_r'], lora_alpha=self.agent_config['lora_alpha'], lora_dropout=self.agent_config['lora_dropout'], bias='none', task_type=TaskType.CAUSAL_LM)
        model = get_peft_model(model, lora_config)
        trainable_parameters = count_trainable_parameters(model)

        dataset = PromptResponseDataset(self.tokenizer, [(build_rewrite_prompt(item.original_text), item.adversarial_text) for item in examples], self.agent_config['max_length'])
        dataloader = DataLoader(dataset, batch_size=self.agent_config['batch_size'], shuffle=True)
        optimizer = AdamW(model.parameters(), lr=self.agent_config['lr'])
        logger = ExperimentLogger(config=self.config, run_name=f'agent_{Path(output_dir).name}', job_type='agent_finetune', tags=['agent', 'biogpt', 'lora'])

        use_amp = self.device.type == 'cuda'
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        epoch_history: List[dict] = []
        for epoch in range(self.agent_config['finetune_epochs']):
            progress = tqdm(dataloader, desc=f'agent finetune {epoch + 1}', leave=False)
            epoch_losses: List[float] = []
            for step, batch in enumerate(progress, start=1):
                batch = {key: value.to(model.device) for key, value in batch.items()}
                with maybe_autocast(use_amp, self.device):
                    outputs = model(**batch)
                    loss = outputs.loss / self.agent_config['grad_accum_steps']
                epoch_losses.append(loss.item() * self.agent_config['grad_accum_steps'])
                scaler.scale(loss).backward()
                if step % self.agent_config['grad_accum_steps'] == 0 or step == len(dataloader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                progress.set_postfix(loss=f'{loss.item() * self.agent_config["grad_accum_steps"]:.4f}')
            epoch_metrics = {'epoch': epoch + 1, 'finetune_loss': float(sum(epoch_losses) / max(len(epoch_losses), 1))}
            epoch_history.append(epoch_metrics)
            logger.log_metrics(epoch_metrics, step=epoch + 1, prefix='agent')

        model.save_pretrained(output_dir)
        self.tokenizer.save_pretrained(output_dir)
        image_paths = {'agent_finetune_curve': plot_training_history(epoch_history, Path(output_dir) / 'agent_finetune_curve.png', title='BioGPT Agent Fine-Tuning')}
        summary = {'output_dir': str(output_dir), 'trainable_parameters': trainable_parameters, 'examples': len(examples), 'epoch_history': epoch_history}
        save_json(summary, Path(output_dir) / 'finetune_summary.json')
        logger.log_metrics({'trainable_parameters': trainable_parameters, 'examples': len(examples)}, prefix='agent_summary')
        logger.log_images_from_paths(image_paths)
        logger.finish()
        LOGGER.info('Agent fine-tuning complete: %s', summary)
        return summary


## Evaluation Pipeline And Adversarial Training Loop

This cell recreates the project’s evaluation entry point and the round-based adversarial loop. It is the highest-level logic in the notebook and orchestrates baseline evaluation, fake generation, hard-sample selection, rewriting, retraining, and round tracking.


In [ ]:
def evaluate_detector_checkpoint(config: dict, model_dir: str | Path, split_path: str | Path, rewrites_path: Optional[str | Path] = None, output_path: Optional[str | Path] = None) -> dict:
    frame = pd.read_csv(resolve_path(split_path))
    threshold = config['evaluation'].get('threshold', 0.5)
    predictor = load_predictor(model_dir=model_dir, max_length=config['detector']['max_length'], batch_size=config['detector']['batch_size'], device_name=config['project'].get('device', 'auto'))
    texts = frame['text'].astype(str).tolist()
    labels = frame['label'].astype(int).tolist()
    scores = predictor.score_texts(texts)
    metrics = compute_classification_metrics(labels, scores, threshold=threshold)
    metrics['num_samples'] = len(frame)
    output_dir = ensure_dir((Path(output_path).parent if output_path else config['evaluation']['metrics_path']))
    prediction_frame = pd.DataFrame(build_prediction_rows(texts, labels, scores, threshold=threshold))
    if config['evaluation'].get('log_predictions', True):
        prediction_frame.to_csv(output_dir / f'{Path(split_path).stem}_predictions.csv', index=False)
    if rewrites_path:
        rewrite_frame = pd.read_csv(resolve_path(rewrites_path))
        if {'original_text', 'rewritten_text', 'detector_confidence'}.issubset(rewrite_frame.columns):
            metrics['evasion_rate'] = compute_evasion_rate(rewrite_frame['detector_confidence'].astype(float).tolist(), evasion_threshold=config['agent']['evasion_threshold'])
            metrics['rewrite_quality'] = compute_rewrite_quality(rewrite_frame['original_text'].astype(str).tolist(), rewrite_frame['rewritten_text'].astype(str).tolist())
    image_paths = {}
    if config['evaluation'].get('save_plots', True):
        image_paths = generate_classification_plots(labels, scores, prediction_frame['predicted_label'].astype(int).tolist(), config['evaluation']['plots_path'], Path(split_path).stem)
    if output_path:
        save_json(metrics, output_path)
    logger = ExperimentLogger(config=config, run_name=f'eval_{Path(model_dir).name}_{Path(split_path).stem}', job_type='evaluation', tags=['evaluation'])
    logger.log_metrics(metrics, prefix='evaluation')
    logger.log_dataframe(f'{Path(split_path).stem}_predictions', prediction_frame)
    logger.log_images_from_paths(image_paths)
    logger.finish()
    LOGGER.info('Evaluation metrics: %s', metrics)
    return metrics

def release_cuda_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def select_hard_samples(fake_texts: List[str], confidences: List[float], high_conf_threshold: float, top_k: int) -> tuple[List[str], List[float]]:
    ranked = sorted(zip(fake_texts, confidences), key=lambda item: item[1], reverse=True)
    selected = [item for item in ranked if item[1] >= high_conf_threshold][:top_k]
    if not selected:
        selected = ranked[:top_k]
    hard_samples = [text for text, _ in selected]
    hard_scores = [score for _, score in selected]
    return hard_samples, hard_scores

def build_augmented_split(train_frame: pd.DataFrame, adversarial_samples: List[str]) -> pd.DataFrame:
    if not adversarial_samples:
        return train_frame.copy()
    adversarial_frame = pd.DataFrame({'title': [''] * len(adversarial_samples), 'text': adversarial_samples, 'label': [1] * len(adversarial_samples)})
    return pd.concat([train_frame, adversarial_frame], ignore_index=True)

def run_adversarial_loop(config: dict, detector_checkpoint: Optional[str | Path] = None, seqgan_checkpoint: Optional[str | Path] = None, agent_checkpoint: Optional[str | Path] = None) -> dict:
    set_seed(config['project']['seed'])
    data_config = config['data']
    loop_config = config['loop']
    detector_config = config['detector']

    train_path = resolve_path(Path(data_config['processed_path']) / data_config['train_file'])
    val_path = resolve_path(Path(data_config['processed_path']) / data_config['val_file'])
    test_path = resolve_path(Path(data_config['processed_path']) / data_config['test_file'])
    train_frame = pd.read_csv(train_path)

    if detector_checkpoint is None:
        detector_checkpoint = resolve_path(detector_config['checkpoint_dir']) / 'baseline'
        if not detector_checkpoint.exists():
            LOGGER.info('Baseline detector checkpoint missing; training baseline detector first.')
            train_detector(config=config, output_dir=detector_checkpoint)

    if seqgan_checkpoint is None:
        seqgan_checkpoint = resolve_path(config['seqgan']['checkpoint_dir'])
        if not (Path(seqgan_checkpoint) / 'seqgan.pt').exists():
            LOGGER.info('SeqGAN checkpoint missing; training SeqGAN first.')
            train_seqgan(config=config, output_dir=seqgan_checkpoint)

    metrics_dir = ensure_dir(config['evaluation']['metrics_path'])
    plots_dir = ensure_dir(config['evaluation']['plots_path'])
    round_data_dir = ensure_dir(loop_config['round_data_dir'])
    logger = ExperimentLogger(config=config, run_name='adversarial_loop', job_type='adversarial_loop', tags=['adversarial-loop', 'robustness'])

    baseline_metrics = evaluate_detector_checkpoint(config=config, model_dir=detector_checkpoint, split_path=test_path, output_path=metrics_dir / 'baseline_metrics.json')
    baseline_auc = baseline_metrics.get('auc', float('nan'))
    logger.log_metrics(baseline_metrics, step=0, prefix='loop')

    history = [baseline_metrics | {'round': 0}]
    current_detector_checkpoint = detector_checkpoint
    current_agent_checkpoint = agent_checkpoint

    for round_index in range(1, loop_config['num_rounds'] + 1):
        LOGGER.info('Starting adversarial round %s', round_index)
        round_dir = ensure_dir(Path(loop_config['round_data_dir']) / f'round_{round_index:02d}')

        fake_texts = generate_fake_texts(checkpoint_dir=seqgan_checkpoint, num_samples=loop_config['fake_pool_size'], device_name=config['project'].get('device', 'auto'))
        detector_scores = score_texts(model_dir=current_detector_checkpoint, texts=fake_texts, config=config)
        release_cuda_memory()

        hard_samples, hard_scores = select_hard_samples(fake_texts, detector_scores, config['agent']['high_conf_threshold'], loop_config['hard_sample_top_k'])

        agent = AdversarialAgent(config=config, checkpoint_dir=current_agent_checkpoint)
        rewritten_samples = agent.rewrite(hard_samples)
        del agent
        release_cuda_memory()

        rewritten_scores = score_texts(model_dir=current_detector_checkpoint, texts=rewritten_samples, config=config)
        release_cuda_memory()

        rewrite_frame = pd.DataFrame({'original_text': hard_samples, 'rewritten_text': rewritten_samples, 'original_detector_confidence': hard_scores, 'detector_confidence': rewritten_scores})
        rewrite_frame.to_csv(round_dir / 'rewrites.csv', index=False)

        augmented_train = build_augmented_split(train_frame, rewritten_samples)
        augmented_train_path = round_dir / 'train_augmented.csv'
        augmented_train.to_csv(augmented_train_path, index=False)

        detector_round_dir = ensure_dir(Path(detector_config['checkpoint_dir']) / f'round_{round_index:02d}')
        train_detector(config=config, train_path=augmented_train_path, val_path=val_path, output_dir=detector_round_dir, init_checkpoint=current_detector_checkpoint)
        current_detector_checkpoint = detector_round_dir
        release_cuda_memory()

        round_metrics = evaluate_detector_checkpoint(config=config, model_dir=current_detector_checkpoint, split_path=test_path, rewrites_path=round_dir / 'rewrites.csv', output_path=metrics_dir / f'round_{round_index:02d}_metrics.json')
        round_metrics['round'] = round_index
        round_metrics['evasion_rate'] = compute_evasion_rate(rewrite_frame['detector_confidence'].astype(float).tolist(), evasion_threshold=config['agent']['evasion_threshold'])
        round_metrics['rewrite_quality'] = compute_rewrite_quality(rewrite_frame['original_text'].astype(str).tolist(), rewrite_frame['rewritten_text'].astype(str).tolist())
        round_metrics.update(compute_confidence_shift(before=rewrite_frame['original_detector_confidence'].astype(float).tolist(), after=rewrite_frame['detector_confidence'].astype(float).tolist()))
        round_metrics['num_rewrites'] = int(len(rewrite_frame))
        round_metrics['successful_evasions'] = int((rewrite_frame['detector_confidence'].astype(float) < config['agent']['evasion_threshold']).sum())
        round_metrics['robustness_delta'] = compute_robustness_delta(round_metrics.get('auc', float('nan')), baseline_auc)
        history.append(round_metrics)
        logger.log_metrics(round_metrics, step=round_index, prefix='loop')
        logger.log_dataframe(f'round_{round_index:02d}_rewrites', rewrite_frame)

        successful_examples = [
            {'original_text': row['original_text'], 'adversarial_text': row['rewritten_text'], 'detector_confidence': row['detector_confidence']}
            for _, row in rewrite_frame.iterrows()
            if row['detector_confidence'] < config['agent']['evasion_threshold']
        ]
        if successful_examples:
            agent = AdversarialAgent(config=config, checkpoint_dir=current_agent_checkpoint)
            current_agent_checkpoint = ensure_dir(Path(config['agent']['checkpoint_dir']) / f'round_{round_index:02d}')
            agent.finetune(
                examples=[SuccessfulEvasionExample(original_text=example['original_text'], adversarial_text=example['adversarial_text'], detector_confidence=example['detector_confidence']) for example in successful_examples],
                output_dir=current_agent_checkpoint,
            )
            del agent
            release_cuda_memory()

        save_json({'history': history}, metrics_dir / 'history.json')
        plot_paths = plot_round_metrics(history, plots_dir)
        logger.log_images_from_paths(plot_paths)

    summary = {'baseline': baseline_metrics, 'history': history}
    logger.finish()
    LOGGER.info('Adversarial loop finished.')
    return summary


## End-To-End Run

This cell runs the complete pipeline from uploaded raw data to final evaluated checkpoint. It is the single top-to-bottom execution cell for the full experiment.


In [ ]:
set_seed(CONFIG['project']['seed'])
save_yaml(CONFIG, 'artifacts/runtime_config.yaml')

data_summary = prepare_data(CONFIG)
processed_dir = resolve_path(CONFIG['data']['processed_path'])
train_path = processed_dir / CONFIG['data']['train_file']
val_path = processed_dir / CONFIG['data']['val_file']
test_path = processed_dir / CONFIG['data']['test_file']

baseline_detector_dir = ensure_dir(Path(CONFIG['detector']['checkpoint_dir']) / 'baseline')
baseline_summary = train_detector(CONFIG, output_dir=baseline_detector_dir)

seqgan_dir = ensure_dir(CONFIG['seqgan']['checkpoint_dir'])
seqgan_summary = train_seqgan(CONFIG, output_dir=seqgan_dir)

loop_summary = run_adversarial_loop(CONFIG, detector_checkpoint=baseline_detector_dir, seqgan_checkpoint=seqgan_dir, agent_checkpoint=None)

detector_checkpoint_root = resolve_path(CONFIG['detector']['checkpoint_dir'])
round_checkpoints = sorted(detector_checkpoint_root.glob('round_*'))
final_detector_dir = round_checkpoints[-1] if round_checkpoints else baseline_detector_dir
final_round_name = final_detector_dir.name if final_detector_dir.name.startswith('round_') else None
final_rewrites_path = None
if final_round_name:
    candidate_rewrites = resolve_path(Path(CONFIG['loop']['round_data_dir']) / final_round_name / 'rewrites.csv')
    if candidate_rewrites.exists():
        final_rewrites_path = candidate_rewrites

final_eval_path = resolve_path(Path(CONFIG['evaluation']['metrics_path']) / 'final_eval_metrics.json')
final_eval_metrics = evaluate_detector_checkpoint(config=CONFIG, model_dir=final_detector_dir, split_path=test_path, rewrites_path=final_rewrites_path, output_path=final_eval_path)

split_overview = []
for split_name, split_path_item in [('train', train_path), ('val', val_path), ('test', test_path)]:
    frame = pd.read_csv(split_path_item)
    split_overview.append({
        'split': split_name,
        'rows': len(frame),
        'real_count': int((frame['label'] == 0).sum()),
        'fake_count': int((frame['label'] == 1).sum()),
        'mean_words': float(frame['text'].astype(str).str.split().str.len().mean()),
    })

seqgan_preview = pd.DataFrame({'generated_fake_text': generate_fake_texts(checkpoint_dir=seqgan_dir, num_samples=ANALYSIS_CONFIG['generated_text_preview_count'], device_name=CONFIG['project']['device'])})

display(pd.DataFrame(split_overview))
display(pd.DataFrame(loop_summary['history']).sort_values('round').reset_index(drop=True).tail())
display(seqgan_preview)
final_eval_metrics


## Result Analysis From Saved Artifacts

This last major analysis cell uses the saved prediction files, metric files, rewrite CSVs, and plots to produce a richer post-run view of the experiment. It adds the extra analysis controls you requested, including threshold sweeps, top errors, rewrite summaries, and a plot gallery.


In [ ]:
def gather_metric_files(metrics_dir: str | Path) -> pd.DataFrame:
    rows = []
    for path in sorted(resolve_path(metrics_dir).glob('*_metrics.json')):
        payload = load_json(path)
        row = {'file': path.name, **payload}
        if path.name == 'baseline_metrics.json':
            row['phase'] = 'baseline'
            row['round'] = 0
        elif path.stem.startswith('round_'):
            row['phase'] = 'adversarial'
            row['round'] = int(path.stem.split('_')[1])
        elif path.name == 'final_eval_metrics.json':
            row['phase'] = 'final_eval'
            row.setdefault('round', math.nan)
        rows.append(row)
    frame = pd.DataFrame(rows)
    if not frame.empty and 'round' in frame.columns:
        frame = frame.sort_values(['phase', 'round'], na_position='last').reset_index(drop=True)
    return frame

def sweep_thresholds(predictions_path: str | Path, thresholds: Sequence[float]) -> pd.DataFrame:
    predictions = pd.read_csv(resolve_path(predictions_path))
    labels = predictions['label'].astype(int).tolist()
    probabilities = predictions['fake_probability'].astype(float).tolist()
    rows = []
    for threshold in thresholds:
        metrics = compute_classification_metrics(labels, probabilities, threshold=float(threshold))
        rows.append({'threshold': float(threshold), **metrics})
    return pd.DataFrame(rows).sort_values('threshold').reset_index(drop=True)

def summarize_prediction_errors(predictions_path: str | Path, top_n: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    predictions = pd.read_csv(resolve_path(predictions_path)).copy()
    predictions['error_type'] = 'correct'
    predictions.loc[(predictions['label'] == 0) & (predictions['predicted_label'] == 1), 'error_type'] = 'false_positive'
    predictions.loc[(predictions['label'] == 1) & (predictions['predicted_label'] == 0), 'error_type'] = 'false_negative'
    false_positives = predictions[predictions['error_type'] == 'false_positive'].sort_values('fake_probability', ascending=False).head(top_n)
    false_negatives = predictions[predictions['error_type'] == 'false_negative'].sort_values('fake_probability', ascending=True).head(top_n)
    return false_positives, false_negatives

def summarize_rewrite_rounds(round_data_dir: str | Path, evasion_threshold: float) -> pd.DataFrame:
    rows = []
    for rewrites_path in sorted(resolve_path(round_data_dir).glob('round_*/rewrites.csv')):
        frame = pd.read_csv(rewrites_path)
        shift_metrics = compute_confidence_shift(before=frame['original_detector_confidence'].astype(float).tolist(), after=frame['detector_confidence'].astype(float).tolist())
        rows.append({'round': rewrites_path.parent.name, 'num_rewrites': int(len(frame)), 'evasion_rate': compute_evasion_rate(frame['detector_confidence'].astype(float).tolist(), evasion_threshold=evasion_threshold), **shift_metrics})
    return pd.DataFrame(rows)

def display_plot_gallery(directories: Sequence[str | Path], max_images: int) -> None:
    plot_paths: List[Path] = []
    for directory in directories:
        plot_paths.extend(sorted(resolve_path(directory).glob('*.png')))
    unique_paths: List[Path] = []
    for plot_path in plot_paths:
        if plot_path not in unique_paths:
            unique_paths.append(plot_path)
    for plot_path in unique_paths[:max_images]:
        print(plot_path)
        display(Image(filename=str(plot_path)))

metrics_dir = resolve_path(CONFIG['evaluation']['metrics_path'])
plots_dir = resolve_path(CONFIG['evaluation']['plots_path'])
metric_files_frame = gather_metric_files(metrics_dir)
history_payload = load_json(metrics_dir / 'history.json')
history_frame = pd.DataFrame(history_payload['history']).sort_values('round').reset_index(drop=True)
final_predictions_path = metrics_dir / f'{test_path.stem}_predictions.csv'
threshold_sweep_frame = sweep_thresholds(final_predictions_path, ANALYSIS_CONFIG['threshold_grid'])
best_threshold_by_f1 = threshold_sweep_frame.sort_values(['f1', 'balanced_accuracy', 'accuracy'], ascending=False).iloc[0]
false_positives, false_negatives = summarize_prediction_errors(final_predictions_path, ANALYSIS_CONFIG['top_error_examples'])
rewrite_round_summary = summarize_rewrite_rounds(CONFIG['loop']['round_data_dir'], CONFIG['agent']['evasion_threshold'])

display(metric_files_frame)
display(history_frame)
display(threshold_sweep_frame)
display(pd.DataFrame([best_threshold_by_f1.to_dict()]))
display(false_positives[['fake_probability', 'text']])
display(false_negatives[['fake_probability', 'text']])
display(rewrite_round_summary)

if final_rewrites_path is not None and final_rewrites_path.exists():
    latest_rewrites = pd.read_csv(final_rewrites_path).sort_values('detector_confidence', ascending=True)
    display(latest_rewrites[['original_detector_confidence', 'detector_confidence', 'original_text', 'rewritten_text']].head(ANALYSIS_CONFIG['top_rewrite_examples']))

history_plot_columns = [column for column in ['auc', 'f1', 'accuracy', 'balanced_accuracy', 'evasion_rate', 'rewrite_quality'] if column in history_frame.columns]
history_plot_frame = history_frame[['round', *history_plot_columns]].melt('round', var_name='metric', value_name='value')
plt.figure(figsize=(12, 6))
sns.lineplot(data=history_plot_frame, x='round', y='value', hue='metric', marker='o')
plt.title('Round-by-Round Metric History')
plt.xlabel('Round')
plt.ylabel('Metric Value')
plt.tight_layout()
plt.show()

threshold_plot_frame = threshold_sweep_frame[['threshold', 'precision', 'recall', 'f1', 'specificity', 'balanced_accuracy']].melt('threshold', var_name='metric', value_name='value')
plt.figure(figsize=(12, 6))
sns.lineplot(data=threshold_plot_frame, x='threshold', y='value', hue='metric', marker='o')
plt.title('Threshold Sweep On Final Test Predictions')
plt.xlabel('Decision Threshold')
plt.ylabel('Metric Value')
plt.tight_layout()
plt.show()

if ANALYSIS_CONFIG['display_saved_plots']:
    display_plot_gallery([CONFIG['evaluation']['plots_path'], baseline_detector_dir, seqgan_dir, final_detector_dir], ANALYSIS_CONFIG['max_saved_plot_images'])


## Final Artifact Summary

This final cell writes a compact summary JSON so the important outputs from the run are easy to find after the notebook finishes.


In [ ]:
artifact_summary = {
    'work_dir': str(WORK_DIR),
    'config_snapshot': str(resolve_path('artifacts/runtime_config.yaml')),
    'data_summary': data_summary,
    'baseline_detector_dir': str(baseline_detector_dir),
    'seqgan_dir': str(seqgan_dir),
    'final_detector_dir': str(final_detector_dir),
    'final_rewrites_path': str(final_rewrites_path) if final_rewrites_path is not None else None,
    'metrics_dir': str(metrics_dir),
    'plots_dir': str(plots_dir),
    'final_eval_path': str(final_eval_path),
    'best_threshold_by_f1': {key: float(value) if isinstance(value, (int, float, np.floating, np.integer)) else value for key, value in best_threshold_by_f1.to_dict().items()},
}

summary_path = resolve_path('artifacts/run_notebook_summary.json')
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open('w', encoding='utf-8') as handle:
    json.dump(artifact_summary, handle, indent=2)

artifact_summary
